In [4]:
# Загружаем нужные библиотеки, добавляю в начало по необходимости
import numpy as np
import tensorflow as tf
import random
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.applications.densenet import preprocess_input, decode_predictions

Задание 1. Загрузите нейросеть архитектуры DenseNet201, предобученную на Imagenet. Прогоните через неё картинку cat.jpg из файлов воркшопа. Как называется наиболее вероятный класс для данной картинки?

In [5]:
# Фикс сидов по ДЗ
np.random.seed(17)
tf.random.set_seed(17)
random.seed(17)

In [6]:
# Загружаем изображение и масштабируем под вход DenseNet201.
# Далее преобразуем в массив, добавляем батч размер и выполняем предобработку
img = image.load_img("cat.jpg", target_size=(224, 224))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

In [7]:
# Загружаем предобученную на ImageNet модель DenseNet201
model = DenseNet201(weights="imagenet")
pred = model.predict(x)

82524592/82524592 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 24s 24s/step


In [8]:
# Декодируем предсказания.
# Мы переводим выход модели (вектора вероятностей) в читаемые названия классов ImageNet.
decode_predictions(pred, top=3)

35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step


[[('n02124075', 'Egyptian_cat', np.float32(0.16923392)),
  ('n01882714', 'koala', np.float32(0.14802769)),
  ('n01883070', 'wombat', np.float32(0.11369926))]]

Задание 2. Выполните transfer-learning этой нейросети на трейновой части того же датасета, что использовался в воркшопе. Заморозьте все слои нейросети, кроме 10 последних. К последнему слою напрямую добавьте ещё один для бинарной классификации.
Сколько тренируемых параметров имеет получившаяся нейросеть?

In [9]:
# Загрузка предобученной денснет
densenet = DenseNet201(weights="imagenet")

In [10]:
# Замораживаем все слои, кроме последних 10 (- от конца)
# Технически мы сначала замораживаем всё, а потом размораживаем последние 10
for layer in densenet.layers:
    layer.trainable = False

for layer in densenet.layers[-10:]:
    layer.trainable = True

In [11]:
# По условию задания меняем последний слой на ReLU, чтобы использовать модель как feature extractor
# Произошло переключение на ReLU потому что нам не нужна нормализация выхода, сумма не должна равняться 1 и т.д.
densenet.layers[-1].activation = tf.keras.activations.relu

In [12]:
# Вся DenseNet это один большой слой. Добавляем классификацию кот/не кот
model_cats = tf.keras.models.Sequential([
    densenet,
    tf.keras.layers.Dense(1, activation="sigmoid")])

In [13]:
model_cats.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet201 (Functional)        │ (None, 1000)           │    20,242,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,001 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,243,985 (77.22 MB)

 Trainable params: 2,204,625 (8.41 MB)

 Non-trainable params: 18,039,360 (68.81 MB)

Задание 3.  Обучите эту нейросеть оптимизатором Adam со стандартными настройками на трейновой части датасета без аугментаций батчами по 16 картинок.

Каково значение F1-метрики на валидации после 50 эпох обучения? Ответ округлите до десятых.

In [21]:
import os

# Загружаем все картинки с папки, применяем preprocess_input для денснет. Формируем массивы x y

def preprocess_image(file):
    img = image.load_img(file, target_size=(224, 224))
    img = image.img_to_array(img)
    img = preprocess_input(img)
    return img

cats = [(preprocess_image('pics/cats/' + f), 1) for f in os.listdir('pics/cats')]
nocats = [(preprocess_image('pics/nocats/' + f), 0) for f in os.listdir('pics/nocats')]

data = cats + nocats
random.shuffle(data)

x = np.array([d[0] for d in data])
y = np.array([d[1] for d in data])

In [22]:
# Начинаем делить данные на train, validation и test (70%/15%/15%)
def split(x, val_frac=0.15, test_frac=0.15):
    n = len(x)
    x_train = x[:int((1 - val_frac - test_frac) * n)]
    x_val = x[int((1 - val_frac - test_frac) * n):int((1 - test_frac) * n)]
    x_test = x[int((1 - test_frac) * n):]
    return x_train, x_val, x_test

x_train, x_val, x_test = split(x)
y_train, y_val, y_test = split(y)

In [23]:
# Для оценки качества модели используются метрики precision и recall, на основе которых вычисляется F1-мера.
# Precision показывает долю корректных положительных предсказаний, recall — долю найденных положительных объектов.
# F1-мера является их гармоническим средним и позволяет оценить баланс между ними.
precision = tf.keras.metrics.Precision()
recall = tf.keras.metrics.Recall()

def f1_metric(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * (p * r) / (p + r + 1e-7)

In [24]:
# На этапе компиляции задаётся способ обучения модели.
# В качестве оптимизатора используется Adam со стандартными параметрами, функция потерь-
# -бинарная кросс-энтропия, так как задача является бинарной классификацией.
# В качестве метрик считаются accuracy, precision, recall и F1.
model_cats.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(),
        precision,
        recall,
        f1_metric])

In [25]:
# Запуск цикла обучения (очень и очень долго)
istory = model_cats.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=50,
    batch_size=16)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 92s 2s/step - binary_accuracy: 0.6687 - f1_metric: 0.5188 - loss: 0.9355 - precision: 0.5977 - recall: 0.6476 - val_binary_accuracy: 0.9275 - val_f1_metric: 0.8982 - val_loss: 0.2124 - val_precision: 0.8214 - val_recall: 1.0000
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - binary_accuracy: 0.9936 - f1_metric: 0.9915 - loss: 0.0241 - precision: 1.0000 - recall: 0.9848 - val_binary_accuracy: 0.9420 - val_f1_metric: 0.9107 - val_loss: 0.2142 - val_precision: 0.8519 - val_recall: 1.0000
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 90ms/step - binary_accuracy: 1.0000 - f1_metric: 1.0000 - loss: 0.0032 - precision: 1.0000 - recall: 1.0000 - val_binary_accuracy: 0.9565 - val_f1_metric: 0.9477 - val_loss: 0.1379 - val_precision: 0.8846 - val_recall: 1.0000
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 97ms/step - binary_accuracy: 1.0000 - f1_metric: 1.0000 - loss: 0.0017 - precision: 1.0000 - recall: 1.0000 - val_binary_accuracy: 0.9565 - val_f1_metric: 0.95